# Sesión 2: Regresión Logística
## Modelo Baseline para Predicción de Resultados

**Objetivo**: Construir nuestro primer modelo predictivo usando Regresión Logística para predecir resultados de partidos de fútbol.

**Duración**: ~60 minutos

**Pregunta**: ¿Podemos predecir si un partido terminará en Victoria Local, Victoria Visitante o Empate?

## 1. Configuración y Carga de Datos

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Librerías de machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# Cargar dataset
data_path = Path('../data/datos_liga_futbol.csv')
df = pd.read_csv(data_path)

print(f"✓ Dataset cargado: {df.shape[0]} partidos")
print(f"✓ Librerías importadas")

## 2. Exploración Rápida

Recordemos las variables clave de nuestro dataset.

In [ ]:
# Ver primeras filas
print("PRIMERAS FILAS DEL DATASET:")
display(df.head())

# Distribución de resultados (nuestra variable objetivo)
print("\nDISTRIBUCIÓN DE RESULTADOS:")
print(df['Resultado'].value_counts())
print(f"\nTotal: {len(df)} partidos")

## 3. Feature Engineering (Ingeniería de Características)

Crearemos nuevas variables (features) que ayuden al modelo a predecir mejor:

1. **Diferencia_Habilidad**: Habilidad_Local - Habilidad_Visitante
2. **Diferencia_Racha**: Racha_Local - Racha_Visitante
3. **Ratio_Habilidad**: Habilidad_Local / Habilidad_Visitante

💡 **Intuición**: Si el equipo local tiene más habilidad y mejor racha, debería ganar más seguido.

In [ ]:
# Crear nuevas features
df['Diferencia_Habilidad'] = df['Habilidad_Local'] - df['Habilidad_Visitante']
df['Diferencia_Racha'] = df['Racha_Local'] - df['Racha_Visitante']
df['Ratio_Habilidad'] = df['Habilidad_Local'] / df['Habilidad_Visitante']

print("✓ Features creados:")
print("  - Diferencia_Habilidad")
print("  - Diferencia_Racha")
print("  - Ratio_Habilidad")

# Ver estadísticas de los nuevos features
print("\nESTADÍSTICAS DE NUEVOS FEATURES:")
print(df[['Diferencia_Habilidad', 'Diferencia_Racha', 'Ratio_Habilidad']].describe())

## 4. Preparación de Datos para Modelado

Dividiremos los datos en:
- **X**: Variables predictoras (features)
- **y**: Variable objetivo (Resultado)
- **Train set**: 80% de los datos para entrenar
- **Test set**: 20% de los datos para evaluar

In [ ]:
# Seleccionar features para el modelo
features = [
    'Habilidad_Local',
    'Habilidad_Visitante',
    'Racha_Local',
    'Racha_Visitante',
    'Diferencia_Habilidad',
    'Diferencia_Racha',
    'Ratio_Habilidad'
]

X = df[features]
y = df['Resultado']

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,  # Para reproducibilidad
    stratify=y  # Mantener proporción de clases
)

print("✓ Datos preparados para modelado")
print(f"\nTamaño del conjunto de entrenamiento: {len(X_train)} partidos ({len(X_train)/len(df)*100:.1f}%)")
print(f"Tamaño del conjunto de prueba: {len(X_test)} partidos ({len(X_test)/len(df)*100:.1f}%)")

# Verificar distribución de clases
print("\nDISTRIBUCIÓN EN TRAIN SET:")
print(y_train.value_counts())
print("\nDISTRIBUCIÓN EN TEST SET:")
print(y_test.value_counts())

## 5. Entrenamiento del Modelo de Regresión Logística

### ¿Qué es Regresión Logística?

La **Regresión Logística** es un algoritmo de clasificación que predice probabilidades de pertenencia a diferentes clases.

**Para nuestro problema**:
- Tenemos 3 clases: Victoria Local, Victoria Visitante, Empate
- El modelo aprende qué combinación de features (habilidad, racha, etc.) llevan a cada resultado
- Usa una función matemática para convertir features en probabilidades

💡 **Analogía**: Es como un experto que mira las estadísticas de ambos equipos y dice "60% probabilidad de victoria local, 30% visitante, 10% empate"

In [ ]:
# Crear y entrenar el modelo
modelo_logistico = LogisticRegression(
    max_iter=1000,  # Iteraciones máximas para convergencia
    random_state=42  # Reproducibilidad
)

# Entrenar el modelo con los datos de entrenamiento
modelo_logistico.fit(X_train, y_train)

print("✓ Modelo entrenado exitosamente")
print(f"\nClases aprendidas: {modelo_logistico.classes_}")
print(f"Número de features utilizados: {len(features)}")

## 6. Predicciones y Evaluación

Ahora usaremos el modelo entrenado para hacer predicciones en el test set.

In [ ]:
# Hacer predicciones en train y test
y_train_pred = modelo_logistico.predict(X_train)
y_test_pred = modelo_logistico.predict(X_test)

# Calcular accuracy (exactitud)
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("RESULTADOS DEL MODELO:")
print(f"\nAccuracy en TRAIN: {train_accuracy:.3f} ({train_accuracy*100:.1f}%)")
print(f"Accuracy en TEST:  {test_accuracy:.3f} ({test_accuracy*100:.1f}%)")

# Interpretar
print(f"\n💡 INTERPRETACIÓN:")
print(f"El modelo acierta {test_accuracy*100:.1f}% de las predicciones en datos nuevos.")
if train_accuracy - test_accuracy > 0.1:
    print("⚠️  Posible overfitting: El modelo funciona mucho mejor en train que en test.")
else:
    print("✓ El modelo generaliza bien a datos nuevos.")

## 7. Matriz de Confusión

La matriz de confusión nos muestra **dónde acierta y dónde falla** el modelo.

In [ ]:
# Calcular matriz de confusión
cm = confusion_matrix(y_test, y_test_pred, labels=modelo_logistico.classes_)

# Visualizar
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=modelo_logistico.classes_, 
            yticklabels=modelo_logistico.classes_,
            cbar_kws={'label': 'Cantidad de partidos'})
plt.title('Matriz de Confusión - Regresión Logística', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Predicción del Modelo', fontsize=12)
plt.ylabel('Resultado Real', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 CÓMO LEER LA MATRIZ:")
print("- Diagonal principal (esquina superior izq a inferior der): Aciertos")
print("- Fuera de la diagonal: Errores del modelo")
print("\nEjemplo: Si en fila 'Empate' y columna 'Victoria Local' hay un 5,")
print("significa que el modelo predijo 'Victoria Local' cuando era 'Empate' 5 veces.")

## 8. Análisis Detallado por Clase

In [ ]:
# Reporte de clasificación
print("REPORTE DE CLASIFICACIÓN:")
print("="*60)
print(classification_report(y_test, y_test_pred, target_names=modelo_logistico.classes_))

# Explicar métricas
print("\n📚 MÉTRICAS EXPLICADAS:")
print("  - Precision: De las predicciones de esta clase, ¿cuántas son correctas?")
print("  - Recall: De todos los casos reales de esta clase, ¿cuántos detectó?")
print("  - F1-score: Promedio armónico de precision y recall")
print("  - Support: Cantidad de casos reales de esta clase en test set")

## 9. Importancia de Features (Coeficientes)

Los coeficientes nos dicen **qué variables son más importantes** para la predicción.

In [ ]:
# Obtener coeficientes
# Nota: En clasificación multiclase, hay un conjunto de coeficientes por cada clase
coeficientes = pd.DataFrame(
    modelo_logistico.coef_.T,
    index=features,
    columns=modelo_logistico.classes_
)

print("COEFICIENTES DEL MODELO:")
print(coeficientes.round(4))

# Visualizar coeficientes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, clase in enumerate(modelo_logistico.classes_):
    coefs = coeficientes[clase].sort_values(ascending=True)
    colors = ['red' if x < 0 else 'green' for x in coefs.values]
    
    axes[idx].barh(range(len(coefs)), coefs.values, color=colors, alpha=0.7)
    axes[idx].set_yticks(range(len(coefs)))
    axes[idx].set_yticklabels(coefs.index)
    axes[idx].axvline(0, color='black', linewidth=0.8)
    axes[idx].set_title(f'Coeficientes para:\n{clase}', fontweight='bold')
    axes[idx].set_xlabel('Valor del Coeficiente')
    axes[idx].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 INTERPRETACIÓN:")
print("  - Coeficientes POSITIVOS (verde): Aumentan la probabilidad de esa clase")
print("  - Coeficientes NEGATIVOS (rojo): Disminuyen la probabilidad de esa clase")
print("  - Valores más grandes (en absoluto): Mayor importancia")

## 10. Ejemplos de Predicciones

Veamos algunos ejemplos concretos de predicciones del modelo.

In [ ]:
# Seleccionar 5 ejemplos del test set
ejemplos_indices = np.random.choice(X_test.index, size=5, replace=False)
ejemplos = df.loc[ejemplos_indices]

# Hacer predicciones
X_ejemplos = ejemplos[features]
predicciones = modelo_logistico.predict(X_ejemplos)
probabilidades = modelo_logistico.predict_proba(X_ejemplos)

print("EJEMPLOS DE PREDICCIONES:")
print("="*80)

for i, idx in enumerate(ejemplos_indices):
    partido = ejemplos.loc[idx]
    pred = predicciones[i]
    probs = probabilidades[i]
    
    print(f"\n📊 PARTIDO {i+1}:")
    print(f"   {partido['Equipo_Local']} (Hab: {partido['Habilidad_Local']}, Racha: {partido['Racha_Local']})")
    print(f"   vs")
    print(f"   {partido['Equipo_Visitante']} (Hab: {partido['Habilidad_Visitante']}, Racha: {partido['Racha_Visitante']})")
    print(f"\n   Resultado Real: {partido['Resultado']}")
    print(f"   Predicción:     {pred}")
    print(f"   {'✓ CORRECTO' if pred == partido['Resultado'] else '✗ INCORRECTO'}")
    print(f"\n   Probabilidades:")
    for j, clase in enumerate(modelo_logistico.classes_):
        print(f"      {clase}: {probs[j]*100:.1f}%")
    print("-" * 80)

## 11. Resumen y Conclusiones

### 📊 Resultados del Modelo Baseline

**Modelo**: Regresión Logística

**Features utilizados** (7):
- Habilidad_Local, Habilidad_Visitante
- Racha_Local, Racha_Visitante
- Diferencia_Habilidad, Diferencia_Racha, Ratio_Habilidad

### 🎯 Métricas Clave

*Nota: Los valores se guardarán después de ejecutar el notebook*

### 💡 Insights

1. **¿Qué aprendimos?**
   - La regresión logística puede predecir resultados de partidos
   - Las diferencias de habilidad y racha son features importantes
   - Algunos resultados son más difíciles de predecir (típicamente los empates)

2. **¿Qué clase predice mejor?**
   - Revisar el classification report arriba
   - Típicamente: Victorias más fáciles de predecir que empates

3. **¿Qué sigue?**
   - En las siguientes sesiones probaremos otros modelos
   - Comparar: ¿Árboles de decisión funcionan mejor?
   - ¿Random Forest o XGBoost mejoran el accuracy?

### 📈 Para Comparación Futura

Guardaremos las métricas para comparar con otros modelos.

In [ ]:
# Guardar métricas para comparación
metricas_logistica = {
    'Modelo': 'Regresión Logística',
    'Accuracy_Train': train_accuracy,
    'Accuracy_Test': test_accuracy,
    'Num_Features': len(features)
}

print("="*70)
print("RESUMEN FINAL - REGRESIÓN LOGÍSTICA")
print("="*70)
print(f"\n✓ Modelo entrenado y evaluado")
print(f"\nAccuracy en Test: {test_accuracy:.3f} ({test_accuracy*100:.1f}%)")
print(f"Features utilizados: {len(features)}")
print(f"\n💾 Métricas guardadas para comparación futura")
print(f"\n🎯 PRÓXIMO PASO: Sesión 3 - Árboles de Decisión")
print("   ¿Podremos mejorar este resultado?")
print("="*70)